# Notebook 02 — Fire Data Processing

**Purpose:** Load raw NASA FIRMS VIIRS CSV files, apply a four-stage filter pipeline (location → season → confidence → date validity), and save the cleaned dataset.

**Input:** `data/raw/fire/*/*.csv`  
**Output:** `data/processed/fire_filtered.csv`

In [8]:
import pandas as pd
import glob

# Load all raw VIIRS CSV files
# NASA FIRMS creates one sub-folder per sensor/tile download.
# The wildcard pattern collects every CSV regardless of sub-folder name.
files = glob.glob('../Data/Raw/Fire/*/*.csv')

print(f"CSV files found: {len(files)}")
if not files:
    raise FileNotFoundError(
        "No fire CSVs found. "
        "Download VIIRS data from firms.modaps.eosdis.nasa.gov and place "
    )

dfs = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    dfs.append(df)
    print(f"  {f}: {len(df):,} rows")

fire = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows (all sensors merged): {len(fire):,}")
print(f"Columns: {fire.columns.tolist()}")


CSV files found: 2
  ../Data/Raw/Fire\DL_FIRE_J1V-C2_725492\fire_archive_J1V-C2_725492.csv: 3,330,822 rows
  ../Data/Raw/Fire\DL_FIRE_SV-C2_725493\fire_archive_SV-C2_725493.csv: 5,392,963 rows

Total rows (all sensors merged): 8,723,785
Columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']


In [9]:
# Inspect the confidence column before filtering 
# VIIRS S-NPP / NOAA-20 use text codes: 'l' (low), 'n' (nominal), 'h' (high).
# MODIS uses numeric 0-100. We need to see which format is present.
print("Unique confidence values:", fire['confidence'].astype(str).unique()[:20])


Unique confidence values: ['n' 'l' 'h']


In [11]:
# Filter pipeline 
print(f"Starting rows: {len(fire):,}")

# Step 1 — Parse dates; drop rows with unparseable dates
fire['acq_date'] = pd.to_datetime(fire['acq_date'], errors='coerce')
before = len(fire)
fire = fire.dropna(subset=['acq_date'])
print(f"After date parse   : {len(fire):,}  (dropped {before - len(fire):,} bad dates)")

# Step 2 — Extract month and year
fire['month'] = fire['acq_date'].dt.month
fire['year']  = fire['acq_date'].dt.year

# Step 3 — Bounding box for Punjab & Haryana
# Lat 27.5–32.5 N, Lon 73.8–77.5 E.  Points outside cannot fall in any district.
lat = fire['latitude'].astype(float)
lon = fire['longitude'].astype(float)
fire = fire[(lat >= 27.5) & (lat <= 32.5) & (lon >= 73.8) & (lon <= 77.5)]
print(f"After location bbox: {len(fire):,}")

# Step 4 — Season filter: keep only stubble-burning windows
# Rabi (wheat harvest) : April–May    → months 4, 5
# Kharif (rice harvest): October–Nov  → months 10, 11
fire = fire[fire['month'].isin([4, 5, 10, 11])]
print(f"After season filter: {len(fire):,}")

# Step 5 — Confidence normalisation and threshold
# Map VIIRS text codes → numeric; MODIS numeric stays as-is.
conf_map = {'l': 10, 'n': 50, 'h': 90}
fire['confidence'] = (
    fire['confidence']
    .astype(str).str.strip().str.lower()
    .replace(conf_map).infer_objects(copy=False)
)
fire['confidence'] = pd.to_numeric(fire['confidence'], errors='coerce')
fire = fire.dropna(subset=['confidence'])
fire = fire[fire['confidence'] >= 30]   # drop 'low' VIIRS detections
print(f"After confidence   : {len(fire):,}  (threshold ≥ 30)")


Starting rows: 1,242,007
After date parse   : 1,242,007  (dropped 0 bad dates)
After location bbox: 1,242,007
After season filter: 1,242,007
After confidence   : 1,242,007  (threshold ≥ 30)


In [12]:
# Save filtered fire data
fire.to_csv('../Data/Processed/fire_filtered.csv', index=False)
print(f"\nSaved: data/processed/fire_filtered.csv")
print(f"   Rows    : {len(fire):,}")
print(f"   Columns : {fire.columns.tolist()}")
print(f"   Years   : {sorted(fire['year'].unique())}")



✅ Saved: data/processed/fire_filtered.csv
   Rows    : 1,242,007
   Columns : ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type', 'month', 'year']
   Years   : [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
